> ⚠️ **作業中 (Work in Progress)**:このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [エージェント概要](#エージェント概要)
- [ModelRouterAgentの作成](#modelrouteragentの作成)
- [FileSearchAgentの作成](#filesearchagentの作成)
- [WebSearchAgentの作成](#websearchagentの作成)
- [エージェントのデプロイと呼び出し](#エージェントのデプロイと呼び出し)

## 🎯 学習目標

- Microsoft Foundryエージェントのコア概念の理解
- Model Routerベースのエージェント構築
- File Search機能を活用したドキュメントベースのエージェント作成
- Web Search機能を活用したリアルタイム情報検索エージェント作成
- エージェントのデプロイとプログラマティック呼び出し方法の学習

## ⏱️ 予想所要時間

約30分

## 環境設定

まずプロジェクトエンドポイントを設定します.

In [ ]:
# 環境変数ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを見つけられるように)
possible_paths = [
  "/opt/homebrew/bin", # macOS (Apple Silicon)
  "/usr/local/bin",   # macOS (Intel) / Linux
  "/usr/bin",      # Linux / GitHub Codespaces
  "/home/linuxbrew/.linuxbrew/bin" # Linux Homebrew
]

az_path = None
try:
  result = subprocess.run(['which', 'az'], capture_output=True, text=True)
  if result.returncode == 0:
    az_path = os.path.dirname(result.stdout.strip())
except:
  pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
  paths_to_add.append(az_path)
else:
  for path in possible_paths:
    if os.path.exists(path) and path not in os.environ.get("PATH", ""):
      paths_to_add.append(path)

if paths_to_add:
  new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
  os.environ["PATH"] = new_path

# が前ノートブックで保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  # 環境変数設定
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  LOCATION = config.get("LOCATION")
  TENANT_ID = config.get("TENANT_ID")
  PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
  PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
  
  # 環境変数でも設定 (他のツールが使用できるように)
  os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
  os.environ["LOCATION"] = LOCATION
  os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
  os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
  os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
  
  print(f"✅ 設定ファイル '{config_file}'で環境変数をロードしました.")
  print(f"\n📌 Foundry Name:{FOUNDRY_NAME}")
  print(f"📌 Resource Group:{RESOURCE_GROUP}")
  print(f"📌 Location:{LOCATION}")
  print(f"📌 プロジェクトエンドポイント:{PROJECT_ENDPOINT}")
  
except FileNotFoundError:
  print(f"⚠️ '{config_file}' ファイルを見つかりません.")
  print("💡 01-setup.ipynbを先に実行して環境を設定してください.")
  raise

# 必須パッケージインストール
%pip install -q azure-ai-projects==2.0.0b2 azure-identity

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント:{PROJECT_ENDPOINT}")

## ModelRouterAgentの作成

Model Routerを活用してインテリジェントなでモデルを選択するはエージェントをだけします.

**Agent 構成:**
- **Model**:model-router (コスト/品質/パフォーマンス自動最適化)
- **Instructions**:質問回答エージェント
- **Tools**:なし (デフォルト会話)

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ModelRouterAgent 作成
ROUTER_INSTRUCTIONS = """あなたは質問に回答するはエージェントです.
リクエストの複雑もと要件にに従ってが章適切なモデルを使用してください.
常に明確で正確でありも動きがなりは回答を提供してください."""

try:
  definition = PromptAgentDefinition(
    model="model-router",
    instructions=ROUTER_INSTRUCTIONS
  )
  
  agent_router = client.agents.create(
    name="ModelRouterAgent",
    definition=definition
  )
  
  print(f"✅ ModelRouterAgent 作成完了!")
  print(f"  ID:{agent_router.id}")
  print(f"  Name:{agent_router.name}")
  print(f"  Model:{definition.model}")
  
except Exception as e:
  print(f"⚠️ エージェント作成失敗:{e}")
  print("\n💡 解決方法:")
  print("  1. Portalで 'model-router' モデルがデプロイされているか確認")
  print("  2. モデル名前が正確か確認 (大文字小文字区別)")

## FileSearchAgentの作成

ファイル検索機能を活用してアップロードされたドキュメントで情報を見つけるはエージェントです.

**Agent 構成:**
- **Model**:gpt-5.1
- **Tools**:file_search (ファイル内容検索)
- **Files**:knowledge-base.json アップロード

**⚠️ 注意**:ファイルアップロードは Azure Portal(https://ai.azure.com)でより便利します.
- Build > Agents > Create agent > Tools > File Search > Upload files

In [ ]:
# FileSearchAgent 作成

FILE_SEARCH_INSTRUCTIONS = """あなたはToolsに登録された File search ベースで回答するはエージェントです.

重要ルール:
1. 必ずアップロードされたファイルの内容を基半でだけ回答してください
2. ファイルにないは情報は "提供されたドキュメントで該当情報を見つかりません"と回答してください
3. 回答時 ソースファイル名を言及してください
4. 正確な引用を使用してください"""

try:
  # FileSearchAgent 作成 (tools ないがまず作成)
  definition = PromptAgentDefinition(
    model="gpt-5.1",
    instructions=FILE_SEARCH_INSTRUCTIONS
  )
  
  agent_filesearch = client.agents.create(
    name="FileSearchAgent",
    definition=definition
  )
  
  print(f"✅ FileSearchAgent 作成完了!")
  print(f"  ID:{agent_filesearch.id}")
  print(f"  Name:{agent_filesearch.name}")
  print(f"  Model:{definition.model}")
  print(f"\n📋 次のステップ:Azure Portalで File Search も旧追加")
  print(f"  1. Azure Portal (https://ai.azure.com) 接続")
  print(f"  2. Build > Agents > 'FileSearchAgent' 選択")
  print(f"  3. Tools セクションで 'File Search' も旧追加")
  print(f"  4. knowledge-base.json ファイルアップロード")
  print(f"  5. アップロード後 エージェントがファイル内容を検索するできるあります")
  
except Exception as e:
  print(f"⚠️ エージェント作成失敗:{e}")
  print("\n💡 解決方法:")
  print("  1. 'gpt-5.1' モデルがデプロイされているか確認")
  print("  2. Portalでモデル名前確認 (大文字小文字区別)")

## WebSearchAgentの作成

Web 検索機能を活用してリアルタイム情報を提供するはエージェントです.

**Agent 構成:**
- **Model**:gpt-4.1
- **Tools**:web_search (Web 検索)
- **機能**:最新ニュース, 天気, 株式情報など

In [ ]:
# WebSearchAgent 作成

WEB_SEARCH_INSTRUCTIONS = """あなたはToolsに登録された Web search ベースで回答するはエージェントです.

重要ルール:
1. 最新情報が必要な質問には必ず Web 検索を使用してください
2. 検索結果を基半で正確で最新の情報を提供してください
3. 回答時 ソース URLを含むしてください
4. 複数のソースの情報を総合してバランスの取れた回答を提供してください
5. 検索結果が不十分なら追加検索を実行してください"""

try:
  definition = PromptAgentDefinition(
    model="gpt-4.1",
    instructions=WEB_SEARCH_INSTRUCTIONS,
    tools=[{"type":"web_search"}]
  )
  
  agent_websearch = client.agents.create(
    name="WebSearchAgent",
    definition=definition
  )
  
  print(f"✅ WebSearchAgent 作成完了!")
  print(f"  ID:{agent_websearch.id}")
  print(f"  Name:{agent_websearch.name}")
  print(f"  Model:{definition.model}")
  print(f"  Tools:Web Search")
  
except Exception as e:
  print(f"⚠️ エージェント作成失敗:{e}")
  print("\n💡 解決方法:")
  print("  1. 'gpt-4.1' モデルがデプロイされているか確認")
  print("  2. Web Searchがプロジェクトで有効化されているか確認")

### 作成されたエージェントリスト確認

## 作成されたエージェント確認

In [ ]:
# すべてのエージェントリスト取得
agents = client.agents.list()

print("=" * 80)
print("作成されたエージェントリスト")
print("=" * 80)

for agent in agents:
  print(f"\n📌 {agent.name}")
  print(f"  ID:{agent.id}")
  
  # versionsで情報抽出
  if 'latest' in agent.versions:
    latest = agent.versions['latest']
    definition = latest.get('definition', {})
    
    model = definition.get('model', 'N/A')
    print(f"  Model:{model}")
    
    tools = definition.get('tools', [])
    if tools:
      tool_types = [t.get('type', 'unknown') if isinstance(t, dict) else str(t) for t in tools]
      print(f"  Tools:{', '.join(tool_types)}")
    else:
      print(f"  Tools:None")

print("\n✅ ポータル確認:https://ai.azure.com > Build > Agents")

## エージェントテスト (選択)

In [ ]:
# エージェントオブジェクト構造確認 (デバッグ用)
print("🔍 agent_router オブジェクトデバッグ:")
print(f"Type:{type(agent_router)}")
print(f"\nAttributes:")
for attr in dir(agent_router):
  if not attr.startswith('_'):
    try:
      value = getattr(agent_router, attr)
      if not callable(value):
        print(f" {attr}:{value}")
    except:
      pass

print(f"\n📋 Raw object:")
print(agent_router)

In [ ]:
# 簡単なエージェントテスト (選択事項)
# ModelRouterAgentと会話

try:
  print("⏳ ModelRouterAgent 実行中...")
  print(f"📍 Agent ID:{agent_router.id}")
  print(f"📍 Agent Name:{agent_router.name}")
  
  # SDK v2 - project clientを通じて OpenAI client 獲得
  openai_client = client.get_openai_client()
  
  # Conversation 作成
  conversation = openai_client.conversations.create()
  print(f"💬 Conversation ID:{conversation.id}")
  
  # Responses APIの呼び出し
  response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent":{"name":agent_router.name, "type":"agent_reference"}},
    input="Pythonでリストをソートする方法を教えてください."
  )
  
  print("\n" + "=" * 80)
  print("🤖 ModelRouterAgent レスポンス:")
  print("=" * 80)
  print(response.output_text)
  
  # Conversation 整理
  openai_client.conversations.delete(conversation_id=conversation.id)
  print("\n✅ テスト成功!")
  
except NameError:
  print("⚠️ agent_router または clientが定義されていません.")
  print("💡 上の環境設定および ModelRouterAgent 作成セルをまず実行してください.")
  
except Exception as e:
  print(f"⚠️ テスト失敗:{e}")
  import traceback
  print(f"\n詳細エラー:\n{traceback.format_exc()}")

In [ ]:
# WebSearchAgent テスト (選択事項)
# 最新情報検索

try:
  print("⏳ WebSearchAgent 実行中 (Web 検索中...)")
  print(f"📍 Agent ID:{agent_websearch.id}")
  print(f"📍 Agent Name:{agent_websearch.name}")
  
  # SDK v2 - project clientを通じて OpenAI client 獲得
  openai_client = client.get_openai_client()
  
  # Conversation 作成
  conversation = openai_client.conversations.create()
  print(f"💬 Conversation ID:{conversation.id}")
  
  # Responses APIの呼び出し
  response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent":{"name":agent_websearch.name, "type":"agent_reference"}},
    input="今日ソウル天気はどのがです?"
  )
  
  print("\n" + "=" * 80)
  print("🌐 WebSearchAgent レスポンス:")
  print("=" * 80)
  print(response.output_text)
  
  # Conversation 整理
  openai_client.conversations.delete(conversation_id=conversation.id)
  print("\n✅ Web 検索も旧がリアルタイム情報をが取得しました!")
  
except NameError:
  print("⚠️ agent_websearch または clientが定義されていません.")
  print("💡 上の環境設定および WebSearchAgent 作成セルをまず実行してください.")
  
except Exception as e:
  print(f"⚠️ テスト失敗:{e}")
  import traceback
  print(f"\n詳細エラー:\n{traceback.format_exc()}")

### ✅ 確認事項

- エージェントが成功的で公開されているか確認
- Python スクリプトがエラー ないが実行なりはない確認
- レスポンスが予想どおりで返却なりはない確認

## 📚 追加リソース

- [Microsoft Foundry Agents 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/overview?view=foundry)
- [Agent SDK ドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/sdk-overview?view=foundry&pivots=programming-language-python)
- [File Search ガイド](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/file-search?view=foundry&pivots=python)
- [Web Search 統合](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/web-search?view=foundry&pivots=python)

## 次のステップ

様々なエージェントをだけ聞いてみました! が第 Foundry IQを使用して高度なナレッジベースを構築してみましょう:

➡️ **[04. Foundry IQ](./04-foundry-iq.ipynb)**:AI Searchと Blob Storageを活用したナレッジベース構築を学習します.